In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

IMG = (128, 128); BS = 32

# Dataset path (extracted folder next to the notebook)
base = "cat&dogs"

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "training_set"), image_size=IMG, batch_size=BS, label_mode="binary")
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "test_set"), image_size=IMG, batch_size=BS, label_mode="binary")

def build(drop=False, l2reg=False):
    reg = l2(.001) if l2reg else None
    layers = [
        tf.keras.layers.Rescaling(1./255),
        Conv2D(32, 3, activation="relu", kernel_regularizer=reg),
        MaxPooling2D(),
        Conv2D(64, 3, activation="relu", kernel_regularizer=reg),
        MaxPooling2D(),
        Flatten(),
        Dense(64, activation="relu", kernel_regularizer=reg)
    ]
    if drop:
        layers.append(Dropout(.5))
    layers.append(Dense(1, activation="sigmoid"))
    m = Sequential(layers)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

# 1. Baseline (no regularization)
print("\n--- Baseline ---")
baseline = build()
baseline.fit(train_ds, validation_data=val_ds, epochs=10)

# 2. Dropout
print("\n--- Dropout ---")
dropout = build(drop=True)
dropout.fit(train_ds, validation_data=val_ds, epochs=10)

# 3. L2 + Early Stopping
print("\n--- L2 + Early Stopping ---")
l2model = build(l2reg=True)
stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
l2model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[stop])